# ATML PA3 Task 3 on Kaggle

1. Import this notebook into Kaggle. Enable **Internet** and select **GPU T4 ×2**. Each T4 handles independent live questions with the original FP16 model and batch size 1. All strategies for a question stay on the same GPU slot.
2. Select **Save Version → Save & Run All** with the GPU enabled. For an interactive run, save the session outputs before stopping it.
3. If the run stops before completion, attach the saved version through **Add Input → Notebook Output Files**, then run the notebook again. Attach only one Task 3 output version at a time. The launcher automatically restores its files and the pipeline checks checkpoint identity and checksums.

Logs, outputs, and checkpoints are written under `/kaggle/working/ATML_PA3_Task3/`. Both workers share a single on-disk model cache. The launcher budgets 12 GB for Task 3, checks for 1.5 GB of additional headroom, and stops at a checkpoint boundary if the working disk runs low. Only saved Kaggle outputs persist across sessions. The five-hour session budget is a safe-stop setting. Total runtime has not been measured on Kaggle.


In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, subprocess, sys, threading, time

ROOT = Path('/kaggle/working/ATML_PA3_Task3')
ROOT.mkdir(parents=True, exist_ok=True)
LOG = ROOT / 'full_pipeline.log'
_log_lock = threading.RLock()
def log(message):
    line = datetime.datetime.now(datetime.timezone.utc).isoformat() + ' ' + message
    with _log_lock:
        print(line, flush=True)
        with LOG.open('a') as f:
            f.write(line + '\n')
            f.flush()

def stream(command, cwd=None, env=None):
    log('[START] ' + ' '.join(map(str, command)))
    done = threading.Event()
    def heartbeat():
        while not done.wait(30):
            log('[HEARTBEAT] Command active: ' + ' '.join(map(str, command[:4])))
    thread = threading.Thread(target=heartbeat, daemon=True)
    thread.start()
    try:
        with subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as child:
            for line in child.stdout:
                log(line.rstrip())
            code = child.wait()
        if code:
            raise RuntimeError('Command failed with exit code ' + str(code) + ': ' + ' '.join(command))
    except BaseException as exc:
        log('[FAIL] ' + repr(exc))
        failed = {'state': 'failed', 'fully_completed': False, 'error': repr(exc)}
        for name in ('status.json', 'final_verification.json'):
            temp = ROOT / (name + '.partial')
            temp.write_text(json.dumps(failed, indent=2))
            temp.replace(ROOT / name)
        raise
    finally:
        done.set()
        thread.join(timeout=2)
    log('[PASS] Command finished; this is not proof of experiment completion')
(ROOT / 'status.json').write_text(json.dumps({'state': 'not run', 'fully_completed': False}))
log('[START] Task 3 launcher. Logs: ' + str(LOG))
import shutil

TASK3_PEAK_BUDGET = 12_000_000_000
KAGGLE_OUTPUT_LIMIT = 20_000_000_000
DISK_RESERVE = 1_500_000_000

def tree_bytes(path):
    return sum(item.stat().st_size for item in path.rglob('*')
               if not item.is_symlink() and item.is_file())

def check_storage(root, incoming=0):
    current = tree_bytes(root)
    other = max(0, tree_bytes(root.parent) - current)
    required = max(TASK3_PEAK_BUDGET - current, incoming)
    free = shutil.disk_usage(root).free
    if other + current + required + DISK_RESERVE > KAGGLE_OUTPUT_LIMIT or free < required + DISK_RESERVE:
        raise RuntimeError(
            f'Insufficient Kaggle space for Task 3: {free / 1e9:.2f} GB free, '
            f'{other / 1e9:.2f} GB of other working files. Use a clean working directory.'
        )
    log(f'[STORAGE] {free / 1e9:.2f} GB free; {current / 1e9:.2f} GB Task 3 data already present.')

def restore_saved_output(root, input_dir):
    """Fill missing files from one attached Task 3 version, preserving current work."""
    candidates = []
    for current, directories, _ in os.walk(input_dir):
        folder = Path(current)
        if folder.name == root.name and any((folder / name).is_dir() for name in ('release', 'runs')):
            candidates.append(folder)
            directories[:] = []
        elif len(folder.relative_to(input_dir).parts) >= 7:
            directories[:] = []
    if len(candidates) > 1:
        raise RuntimeError('Attach only the Task 3 output version you want to resume.')
    if not candidates:
        log('[START] No attached Task 3 output version found.')
        check_storage(root)
        return
    source = candidates[0]
    log('[START] Restoring from ' + str(source))
    copies = []
    for name in ('release', 'runs', 'model_cache'):
        parent = source / name
        if not parent.is_dir():
            continue
        for current, directories, files in os.walk(parent):
            directories[:] = [directory for directory in directories
                              if directory not in ('staging', '.locks')]
            for filename in files:
                path = Path(current) / filename
                if filename.endswith(('.partial', '.incomplete', '.lock')) or not path.is_file():
                    continue
                target = root / path.relative_to(source)
                if target.exists():
                    continue
                copies.append((path, target))
    check_storage(root, incoming=sum(path.stat().st_size for path, _ in copies))
    for path, target in copies:
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = target.with_name(target.name + '.restore.partial')
        shutil.copyfile(path, temporary)
        temporary.replace(target)
    log('[PASS] Attached files restored. The worker will validate them before reuse.')

restore_saved_output(ROOT, Path('/kaggle/input'))


In [ ]:
SOURCE_REVISION = 'b370e572a01f5b0c3cd4d12e0195314be647093f'
REPOSITORY = 'https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning.git'
CODE = Path('/tmp/atml_task3_' + SOURCE_REVISION[:12])
CODE.mkdir(parents=True, exist_ok=True)
stream(['git', 'init', str(CODE)])
stream(['git', '-C', str(CODE), 'fetch', '--depth=1', '--filter=blob:none', REPOSITORY, SOURCE_REVISION])
stream(['git', '-C', str(CODE), 'sparse-checkout', 'set', '--no-cone',
        '/Assignment-3/task3/', '/Assignment-3/tests/test_task3.py', '/Assignment-3/ATML-PA3.pdf'])
stream(['git', '-C', str(CODE), 'checkout', '--detach', 'FETCH_HEAD'])
stream(['git', '-C', str(CODE), 'diff', '--exit-code', 'HEAD', '--', 'Assignment-3/task3', 'Assignment-3/tests/test_task3.py'])
ASSIGNMENT = CODE / 'Assignment-3'
ENV = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONPATH=str(ASSIGNMENT),
           HF_HOME=str(ROOT / 'model_cache'), TASK3_NOTEBOOK_TEE='1', TOKENIZERS_PARALLELISM='false')
log('[PASS] Pinned source commit: ' + SOURCE_REVISION)
# Workers will each narrow this allocation to one T4 before importing PyTorch.
visible = os.environ.get('CUDA_VISIBLE_DEVICES', '0,1')
ENV['CUDA_VISIBLE_DEVICES'] = visible
ENV['TASK3_T4_DEVICES'] = ','.join(device.strip() for device in visible.split(','))
ENV['HF_HUB_DISABLE_XET'] = '1'
ENV['PIP_NO_CACHE_DIR'] = '1'
stream([sys.executable, '-u', '-c', """
import torch
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
if len(names) != 2 or not all('T4' in name for name in names):
    raise RuntimeError(f'Enable GPU T4 x2 in Kaggle settings. Detected {names}.')
print(f'[PASS] Both GPUs available: {names}; installed PyTorch {torch.__version__}')
"""], cwd=str(ASSIGNMENT), env=ENV)
import importlib.metadata as metadata

missing = []
for requirement in (ASSIGNMENT / 'task3/requirements-colab.txt').read_text().splitlines():
    if not requirement or requirement.startswith('#'):
        continue
    package, version = requirement.split('==', 1)
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(requirement)
if missing:
    constraint = CODE / 'torch-constraint.txt'
    constraint.write_text('torch==' + metadata.version('torch') + '\n')
    stream([sys.executable, '-u', '-m', 'pip', 'install', '--disable-pip-version-check',
            '--prefer-binary', '-c', str(constraint), *missing], env=ENV)
else:
    log('[SKIP] All pinned dependencies are already installed.')
stream([sys.executable, '-u', '-m', 'pytest', '-q', 'tests/test_task3.py'],
       cwd=str(ASSIGNMENT), env=ENV)


In [ ]:
def prepare_saved_output(root):
    """Materialize Hugging Face snapshot links without keeping duplicate model blobs."""
    from hashlib import file_digest

    model_cache = root / 'model_cache'
    for manifest_file in model_cache.glob('*-verified.json'):
        manifest = json.loads(manifest_file.read_text())
        model_dir = model_cache / ('models--' + manifest['model'].replace('/', '--'))
        snapshot = model_dir / 'snapshots' / manifest['revision']
        blob_dir = (model_dir / 'blobs').resolve()
        links = []
        for filename, expected in manifest['files'].items():
            path = snapshot / filename
            if not path.is_symlink():
                continue
            blob = path.resolve()
            if not blob.is_relative_to(blob_dir) or not blob.is_file():
                raise RuntimeError('Invalid cached model link: ' + str(path))
            with blob.open('rb') as handle:
                if file_digest(handle, 'sha256').hexdigest() != expected:
                    raise RuntimeError('Cached model checksum mismatch: ' + str(path))
            links.append((path, blob))
        for path, blob in links:
            temporary = path.with_name(path.name + '.portable.partial')
            temporary.unlink(missing_ok=True)
            os.link(blob, temporary)
            temporary.replace(path)
        for blob in {blob for _, blob in links}:
            # Keep blobs still referenced by any other snapshot in this cache.
            if not any(link.is_symlink() and link.resolve() == blob
                       for link in (model_dir / 'snapshots').rglob('*')):
                blob.unlink()
        if links:
            log('[PASS] Verified model snapshot prepared as regular Kaggle output files.')
    # These are reconstructible copies, never the release, exports, or checkpoints.
    for temporary in [root / 'local_ssd', *root.glob('runs/*/staging')]:
        if temporary.is_dir() and not temporary.is_symlink():
            shutil.rmtree(temporary)

DRIVER = CODE / 'task3_kaggle_driver.py'
DRIVER.write_text('"""Run the pinned Task 3 protocol on Kaggle with one live worker per T4."""\n\nfrom collections import deque\nfrom contextlib import contextmanager\nimport fcntl\nimport json\nimport multiprocessing as mp\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nimport time\nimport urllib.request\n\nfrom task3 import run\nfrom task3.data import QuestionReader\nfrom task3.live import CALLS, Generator, run_live\nfrom task3.storage import Monitor, SafeStop, Store, sha256, write_json\n\nSAFE_STOP_EXIT = 75\nDISK_RESERVE = 1_500_000_000\nMODEL_CACHE_BYTES = 6_183_451_098\nofficial_download = run.download\nofficial_checkpoint_exports = run.checkpoint_exports\n\n\nclass KaggleMonitor(Monitor):\n    """Stop at an existing checkpoint boundary if the working disk gets too full."""\n\n    def check(self, estimated_seconds=0):\n        super().check(estimated_seconds)\n        now = time.monotonic()\n        if now >= getattr(self, "next_disk_check", 0):\n            if shutil.disk_usage(self.root).free < DISK_RESERVE:\n                raise SafeStop("Kaggle working disk below 1.5 GB free; save this output version")\n            self.next_disk_check = now + 15\n\n\nclass WorkerMonitor(KaggleMonitor):\n    def __init__(self, root, deadline, stop, slot):\n        self.stop = stop\n        self.slot = slot\n        super().__init__(root)\n        self.deadline = deadline\n\n    def check(self, estimated_seconds=0):\n        if self.stop.is_set():\n            raise SafeStop("Other GPU worker stopped; completed calls are saved")\n        super().check(estimated_seconds)\n\n    def log(self, kind, message, **fields):\n        if "question" in fields and hasattr(self, "question"):\n            fields["question"] = self.question\n        super().log(kind, f"T4 worker {self.slot}: {message}", **fields)\n\n\ndef check_model_space(cache):\n    cached = sum(path.stat().st_size for path in cache.rglob("*")\n                 if path.is_file() and not path.is_symlink())\n    required = max(0, MODEL_CACHE_BYTES - cached) + DISK_RESERVE\n    if shutil.disk_usage(cache).free < required:\n        raise SafeStop("Insufficient space to finish the shared model cache; save this output version")\n\n\nclass WorkerStore(Store):\n    """Keep each worker\'s smoke-test writes separate from the parent marker."""\n\n    def __init__(self, root, identity, slot):\n        super().__init__(root, identity)\n        self.smoke_key = f"gpu_smoke_t4_{slot}"\n\n    def put(self, key, value):\n        return super().put(self.smoke_key if key == "gpu_smoke" else key, value)\n\n    def get(self, key):\n        return super().get(self.smoke_key if key == "gpu_smoke" else key)\n\n\n@contextmanager\ndef model_load_lock(cache, monitor):\n    cache.mkdir(parents=True, exist_ok=True)\n    with (cache / ".task3-model-load.lock").open("a") as handle:\n        while True:\n            monitor.check()\n            try:\n                fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)\n                break\n            except BlockingIOError:\n                time.sleep(1)\n        try:\n            yield\n        finally:\n            fcntl.flock(handle, fcntl.LOCK_UN)\n\n\ndef pending_work(ids, store):\n    """Resume partial questions on their assigned GPU slot, queue fresh work jointly."""\n    assigned = [[], []]\n    fresh = []\n    for qid in ids:\n        present = [store.get(f"live/{qid}/{name}") is not None for name, _ in CALLS]\n        if all(present) and store.get(f"live_summary/{qid}") is not None:\n            continue\n        affinity = store.get(f"gpu_affinity/{qid}")\n        if affinity is not None:\n            assigned[affinity["slot"]].append(qid)\n        elif any(present):\n            # Partial questions from the original single-GPU notebook used slot zero.\n            assigned[0].append(qid)\n        else:\n            fresh.append(qid)\n    return assigned, fresh\n\n\ndef gpu_worker(slot, device, folder, ids, queue, assigned, checkpoints, identity,\n               cache, logs, deadline, stop):\n    # Set this before Generator or any model code imports CUDA PyTorch.\n    os.environ["CUDA_VISIBLE_DEVICES"] = device\n    monitor = WorkerMonitor(logs / f"t4-{slot}", deadline, stop, slot)\n    store = WorkerStore(checkpoints, identity, slot)\n    try:\n        reader = QuestionReader(folder, "math")\n        generator = None\n\n        class SessionGenerator(Generator):\n            def smoke(self, questions, destination):\n                if not getattr(self, "smoke_complete", False):\n                    questions = [reader.question(qid)[0]["question_text"] for qid in ids]\n                    super().smoke(questions, destination)\n                    self.smoke_complete = True\n\n        def get_generator(model_cache, live_monitor):\n            nonlocal generator\n            if generator is None:\n                with model_load_lock(model_cache, live_monitor):\n                    check_model_space(model_cache)\n                    generator = SessionGenerator(model_cache, live_monitor)\n            return generator\n\n        remaining = deque(assigned)\n        while True:\n            monitor.check()\n            qid = remaining.popleft() if remaining else queue.get()\n            if qid is None:\n                break\n            if store.get(f"gpu_affinity/{qid}") is None:\n                store.put(f"gpu_affinity/{qid}", {"slot": slot})\n            monitor.question = ids.index(qid) + 1\n            monitor.log("START", "Live question assigned", question=monitor.question, qid=qid)\n            run_live(reader, [qid], store, monitor, cache, generator_factory=get_generator)\n    except SafeStop as exc:\n        stop.set()\n        monitor.log("SAFE STOP", str(exc))\n        raise SystemExit(SAFE_STOP_EXIT) from None\n    except BaseException:\n        stop.set()\n        raise\n    finally:\n        monitor.close()\n\n\ndef run_live_dual(reader, ids, store, monitor, model_cache):\n    """Distribute questions; the original generator, seeds, and exports stay intact."""\n    assigned, fresh = pending_work(ids, store)\n    if not (fresh or any(assigned)):\n        monitor.log("SKIP: verified cache", "All live calls and summaries already saved")\n        return\n\n    devices = os.environ["TASK3_T4_DEVICES"].split(",")\n    if len(devices) != 2 or len(set(devices)) != 2 or not all(devices):\n        raise RuntimeError("Two distinct T4 device IDs are required")\n    context = mp.get_context("spawn")\n    queue = context.Queue()\n    stop = context.Event()\n    logs = store.root.parent / "gpu-workers"\n    for qid in [*fresh, None, None]:\n        queue.put(qid)\n    workers = []\n    started = time.monotonic()\n    try:\n        for slot, device in enumerate(devices):\n            worker = context.Process(\n                target=gpu_worker,\n                args=(slot, device, reader.folder, ids, queue, assigned[slot], store.root,\n                      store.identity, Path(model_cache), logs, monitor.deadline, stop),\n            )\n            worker.start()\n            workers.append(worker)\n        while any(worker.is_alive() for worker in workers):\n            if any(worker.exitcode not in (None, 0) for worker in workers):\n                stop.set()\n            for worker in workers:\n                worker.join(timeout=0.25)\n    finally:\n        stop.set()\n        for worker in workers:\n            worker.join(timeout=5)\n            if worker.is_alive():\n                worker.terminate()\n                worker.join()\n        queue.cancel_join_thread()\n        queue.close()\n\n    codes = [worker.exitcode for worker in workers]\n    smoked = [slot for slot in range(2) if store.get(f"gpu_smoke_t4_{slot}") is not None]\n    if smoked:\n        store.put("gpu_smoke", {"mode": "Kaggle T4 workers", "verified_worker_slots": smoked})\n    write_json(logs / "session.json", {\n        "workers": codes,\n        "verified_worker_slots": smoked,\n        "session_worker_wall_seconds": time.monotonic() - started,\n    })\n    if any(code not in (0, SAFE_STOP_EXIT) for code in codes):\n        raise RuntimeError(f"GPU worker failed (exit codes {codes}); completed calls remain saved")\n    if SAFE_STOP_EXIT in codes:\n        raise SafeStop("Live workers stopped at the session or disk safety boundary")\n\n\ndef download_release(root, monitor):\n    """Recover transient public-release download errors using the same file IDs."""\n    from gdown.exceptions import FileURLRetrievalError\n\n    try:\n        return official_download(root, monitor)\n    except (OSError, FileURLRetrievalError) as exc:\n        monitor.log("RETRY", "Trying the direct public-release endpoint", error=repr(exc))\n    manifest_path = Path(run.__file__).with_name("release_manifest.json")\n    manifest = json.loads(manifest_path.read_text())\n    for filename, entry in manifest.items():\n        monitor.check()\n        destination = root / filename\n        if destination.is_file() and sha256(destination) == entry["sha256"]:\n            continue\n        source = entry["source"]\n        url = source if source.startswith("https:") else (\n            f"https://drive.usercontent.google.com/download?id={source}&export=download&confirm=t"\n        )\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        partial = destination.with_name(destination.name + ".direct.partial")\n        try:\n            with urllib.request.urlopen(url, timeout=60) as response, partial.open("wb") as output:\n                remaining = entry["bytes"]\n                while remaining:\n                    block = response.read(min(8 * 1024 * 1024, remaining))\n                    if not block:\n                        break\n                    output.write(block)\n                    remaining -= len(block)\n                    monitor.check()\n                if remaining or response.read(1):\n                    raise IOError(f"Unexpected release file size: {filename}")\n            if sha256(partial) != entry["sha256"]:\n                raise IOError(f"Official release checksum mismatch: {filename}")\n            partial.replace(destination)\n        finally:\n            partial.unlink(missing_ok=True)\n    write_json(root / "release.json", manifest)\n    return official_download(root, monitor)\n\n\ndef checkpoint_exports(store, key, out, names):\n    official_checkpoint_exports(store, key, out, names)\n    staging = out.parent.parent / "staging" / out.name\n    if staging.is_dir():\n        shutil.rmtree(staging)\n\n\ndef main():\n    run.run_live = run_live_dual\n    run.Monitor = KaggleMonitor\n    run.download = download_release\n    run.checkpoint_exports = checkpoint_exports\n    run.main()\n\n\nif __name__ == "__main__":\n    main()\n')
release = ROOT / 'release'
release.mkdir(exist_ok=True)
local_release = ROOT / 'local_ssd'
if not local_release.exists() and not local_release.is_symlink():
    local_release.symlink_to(release, target_is_directory=True)
try:
    log('[START] Offline analysis, then live questions on two T4 workers.')
    stream([sys.executable, '-u', str(DRIVER), '--root', str(ROOT)], cwd=str(ASSIGNMENT), env=ENV)
finally:
    if local_release.is_symlink() and local_release.resolve() == release:
        local_release.unlink()
    prepare_saved_output(ROOT)
    log(f'[STORAGE] Task 3 saved output: {tree_bytes(ROOT) / 1e9:.2f} GB.')
    log('[KAGGLE OUTPUT] Save this version with output files before ending the session.')


In [ ]:
verification = ROOT / 'final_verification.json'
if not verification.is_file():
    raise RuntimeError('Missing final_verification.json: completion cannot be claimed.')
result = json.loads(verification.read_text())
if result.get('state') == 'failed':
    raise RuntimeError(result.get('error', 'Pipeline failed'))
if result.get('fully_completed'):
    required_counts = {'offline_datasets': 2, 'live_questions': 50,
                       'live_generation_calls': 250, 'adaptive_policies': 4}
    for name, count in required_counts.items():
        if result.get(name) != count:
            raise RuntimeError('Incomplete count: ' + name)
    artifacts = result.get('artifacts', {})
    if len(artifacts) < 60 or result.get('artifact_count') != len(artifacts):
        raise RuntimeError('Missing required artifact inventory')
    for filename, expected in artifacts.items():
        path = Path(filename)
        if not path.is_file():
            raise RuntimeError('Missing artifact: ' + filename)
        h = hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
                h.update(block)
        if h.hexdigest() != expected:
            raise RuntimeError('Artifact checksum mismatch: ' + filename)
    log('[COMPLETE] TASK 3 FULLY COMPLETED')
else:
    log('[SAFE STOP] INCOMPLETE — SAVE KAGGLE OUTPUTS AND ATTACH THEM TO RESUME')
log('State: ' + result.get('state', 'unknown'))
log('Offline datasets: ' + str(result.get('offline_datasets', 0)) + '/2')
log('Live questions: ' + str(result.get('live_questions', 0)) + '/50; generation calls: '
    + str(result.get('live_generation_calls', 0)) + '/250')
log('Adaptive policies: ' + str(result.get('adaptive_policies', 0)) + '/4; plots: '
    + str(result.get('plots', 0)) + '; tables: ' + str(result.get('tables', 0))
    + '; total artifacts: ' + str(result.get('artifact_count', 0)))
log('Latest checkpoint: ' + str(result.get('latest_checkpoint', ROOT / 'latest_run.json')))
log('Kaggle output path: ' + str(ROOT))
log('fully_completed = ' + str(result.get('fully_completed', False)))
